# Softmax·Temperature·Top-k/p 샘플링 실습 - Temperature and Nucleus Sampling

- Tutorial ID: `mod-softmax-temperature-lab`
- Tutorial: Softmax·Temperature·Top-k/p 샘플링 실습
- Section ID: `sampling-1`
- Section: Temperature and Nucleus Sampling


In [ ]:
# ============================================================
# 코드 읽는 법 — Temperature and Nucleus Sampling
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
logits = np.array([5.0, 4.2, 3.1, 1.0, 0.5, -0.2])
vocab = ['A','B','C','D','E','F']
def softmax(z):
    e = np.exp(z - z.max()); return e/e.sum()
for T in [0.5, 1.0, 2.0]:
    p = softmax(logits/T)
    H = -np.sum(p*np.log2(p))
        print('T=',T,'probs=',dict(zip(vocab, np.round(p,3))),'entropy=',round(H,3))
p = softmax(logits)
order = np.argsort(-p); cum = np.cumsum(p[order])
top_p = order[cum <= 0.9].tolist();
if len(top_p)==0 or cum[len(top_p)-1] < 0.9: top_p.append(order[len(top_p)])
print('top-p nucleus:', [vocab[i] for i in top_p])